# Module 3: Content Understanding - AI-Powered Document Analysis

## 🔄 Building on Module 2

In **Module 2**, we solved the structure problem with Document Intelligence:
- ✅ Tables → Preserved rows/columns
- ✅ Figures → Detected with bounding boxes
- ✅ Paragraphs → Identified roles (header, footer, content)

**But there's still a gap:**

| What DI Gives Us | What RAG Needs |
|------------------|----------------|
| Figure at `[x, y, width, height]` | **Description**: "A zoning map showing residential areas in pink..." |
| Chart bounding box | **Data**: The actual values and trends |
| Diagram location | **Explanation**: What the diagram represents |

**Content Understanding bridges this gap** using GPT-4.1-mini to generate semantic descriptions automatically.

## 🎯 Learning Objectives

By the end of this module, you will:
1. Understand what **Content Understanding** adds beyond Document Intelligence
2. Use the `prebuilt-documentSearch` analyzer for RAG-optimized extraction
3. See how figures get **AI-generated descriptions** (the "invisible" becomes "describable")
4. Explore the **complete markdown output** with embedded figures
5. Understand how CU prepares content for chunking (covered in Module 4)

---

## Step 0: Let's See What We're Working With

Before we start, let's look at **Page 1 of `metro-s36.pdf`** - the same document from Modules 1 and 2.

This page has:
- 🗺️ A **zoning map** (800m radius around Station 36)
- 📷 Three **street photos** showing different views
- 📊 A **legend** with color-coded land use categories

**In Module 1**: These were completely invisible to our RAG pipeline.  
**In Module 2**: We detected their locations (bounding boxes).  
**In Module 3**: We'll get **AI descriptions** of what's IN them!

In [ ]:
# First, let's render Page 1 of our PDF so you can see what we're analyzing
# This requires pdf2image (optional - will show placeholder if not available)

from pathlib import Path
from IPython.display import display, Image, Markdown, HTML
import os

DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"

print(f"📄 Target Document: {PDF_PATH.name}")
print(f"   This is the SAME document from Modules 1 and 2!\n")

# Try to render the first page as an image
try:
    from pdf2image import convert_from_path
    
    print("🖼️ Rendering Page 1 for reference...")
    pages = convert_from_path(PDF_PATH, first_page=1, last_page=1, dpi=150)
    
    # Save temporarily for display
    page1_path = "page1_preview.png"
    pages[0].save(page1_path, "PNG")
    
    print("\n👀 HERE'S WHAT WE'RE ANALYZING (Page 1):")
    print("="*60)
    display(Image(filename=page1_path, width=600))
    print("="*60)
    print("\n🎯 Notice:")
    print("   - The MAP with colored zones (מגורים, תעסוקה, מסחר, etc.)")
    print("   - The PHOTOS showing street views")
    print("   - The LEGEND explaining the color codes")
    print("\n   Content Understanding will DESCRIBE all of these!")
    
except ImportError:
    print("⚠️ pdf2image not installed. Showing text description instead.")
    print("\n📄 Page 1 of metro-s36.pdf contains:")
    print("   ┌─────────────────────────────────────────────────────┐")
    print("   │  🗺️ ZONING MAP           │  📋 STATION INFO        │")
    print("   │  (800m radius)           │  - מיקום התחנה          │")
    print("   │  Colors show land use:   │  - סוג תחנה             │")
    print("   │  - Pink = Residential    │  - קיבולת נוסעים        │")
    print("   │  - Blue = Commercial     │                         │")
    print("   │                          │                         │")
    print("   ├──────────┬──────────┬────┴─────────────────────────┤")
    print("   │  📷 East │  📷 South│  📷 North                    │")
    print("   │  View    │  View    │  View                        │")
    print("   └──────────┴──────────┴──────────────────────────────┘")
    print("\n   💡 To see the actual PDF, open: data/sample-pdfs/metro-s36.pdf")

---

## Step 1: Setup - Initialize Clients

We need to set up:
1. **Content Understanding Client** - For semantic analysis with `prebuilt-documentSearch`
2. **Credentials** - Using Entra ID (same as Module 2)

### 🔑 Key Difference: Content Understanding vs Document Intelligence

| Aspect | Document Intelligence | Content Understanding |
|--------|----------------------|----------------------|
| **Endpoint** | `*.cognitiveservices.azure.com` | `*.services.ai.azure.com` |
| **Figure Output** | Bounding box only | Bounding box + **AI description** |
| **Requires LLM** | No | Yes (GPT-4.1-mini) |
| **Cost** | Lower | Higher (LLM calls) |

In [ ]:
# Install the Content Understanding SDK if needed
%pip install azure-ai-contentunderstanding -q

In [ ]:
import os
import sys
import json
import re
from pathlib import Path

# Add src to path for shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
from IPython.display import display, Image, Markdown, HTML
from azure.identity import DefaultAzureCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

# Load environment variables
env = load_env()

# --- Setup Credentials ---
print("🔐 Initializing with DefaultAzureCredential (Entra ID)...")
credential = DefaultAzureCredential()

# --- Setup Content Understanding Client ---
# CU uses a different endpoint format than Document Intelligence
doc_endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
cu_endpoint = doc_endpoint.replace(".cognitiveservices.azure.com", ".services.ai.azure.com")

print(f"\n📍 Endpoints:")
print(f"   Document Intelligence: {doc_endpoint}")
print(f"   Content Understanding: {cu_endpoint}")

# Initialize client with GA API version
API_VERSION = "2025-11-01"  # GA version
cu_client = ContentUnderstandingClient(
    endpoint=cu_endpoint,
    credential=credential,
    api_version=API_VERSION
)

# Define paths
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"

print(f"\n✅ Content Understanding Client Ready (API: {API_VERSION})")
print(f"   Target document: {PDF_PATH.name}")

---

## Step 2: Configure Model Deployments

Content Understanding's `prebuilt-documentSearch` analyzer uses **GPT-4.1-mini** internally to:
- Generate semantic descriptions for figures
- Convert charts to Chart.js code
- Convert diagrams to Mermaid.js syntax

We need to tell CU which Azure OpenAI deployment to use.

### ⚠️ Important: Model Mapping Required

CU expects specific model names mapped to your Azure OpenAI deployments:

| CU Model Name | Your Deployment Name |
|---------------|---------------------|
| `gpt-4.1-mini` | (from your Azure OpenAI resource) |
| `text-embedding-3-large` | (optional, for embeddings) |

In [ ]:
# Configure Content Understanding to use your Azure OpenAI deployments
# This is a ONE-TIME setup per resource

print("⚙️ Configuring Content Understanding Model Deployments...\n")

# Get deployment names from environment
gpt41_deployment = env.get("GPT_4_1_DEPLOYMENT", "gpt-4.1")
gpt41_mini_deployment = env.get("GPT_4_1_MINI_DEPLOYMENT", "gpt-4.1-mini")
embedding_deployment = env.get("TEXT_EMBEDDING_3_LARGE_DEPLOYMENT", "text-embedding-3-large")

print(f"   📋 Model deployments to configure:")
print(f"      gpt-4.1 → {gpt41_deployment}")
print(f"      gpt-4.1-mini → {gpt41_mini_deployment}  ← Required for prebuilt-documentSearch")
print(f"      text-embedding-3-large → {embedding_deployment}")

try:
    # Check current configuration
    print("\n   🔎 Checking current configuration...")
    try:
        current_defaults = cu_client.get_defaults()
        if current_defaults.model_deployments:
            print(f"   Current: {current_defaults.model_deployments}")
        else:
            print("   No defaults configured yet.")
    except Exception:
        # get_defaults() fails with DefaultsNotSet when no defaults exist yet — that's fine,
        # we'll set them below via update_defaults()
        print("   No defaults set yet — will configure now.")

    # Update with our deployment mapping
    model_deployments = {
        "gpt-4.1": gpt41_deployment,
        "gpt-4.1-mini": gpt41_mini_deployment,
        "text-embedding-3-large": embedding_deployment,
    }

    print("\n   🔧 Updating defaults...")
    updated_defaults = cu_client.update_defaults(model_deployments=model_deployments)
    print("   ✅ Defaults configured successfully!")

    if updated_defaults.model_deployments:
        print("\n   Final configuration:")
        for model_name, deployment_name in updated_defaults.model_deployments.items():
            print(f"      {model_name}: {deployment_name}")

except Exception as e:
    print(f"\n❌ Configuration failed: {e}")
    print("\n   ⚠️ TROUBLESHOOTING:")
    print("      1. Ensure you have 'Cognitive Services User' role on the resource")
    print("      2. Verify model deployments exist in Azure AI Foundry")
    print("      3. Check that deployment names in .env match your Azure deployments")

---

## Step 3: Analyze Document with Content Understanding

Now for the magic! We'll use the `prebuilt-documentSearch` analyzer which:

1. **Extracts text** with reading order (like DI)
2. **Detects figures** and outputs markdown image references (e.g., `![alt](figures/1.1)`)
3. **Generates AI descriptions** for each figure (NEW! — this is what DI doesn't do)
4. **Converts charts → Chart.js** and **diagrams → Mermaid.js** (NEW!)
5. **Outputs GitHub Flavored Markdown** optimized for LLMs

"> ⚠️ **Note**: CU outputs both bounding box coordinates (polygon geometry via the `source` field) **and** AI-generated semantic descriptions for figures. The `prebuilt-documentSearch` analyzer provides bounding boxes plus AI descriptions, Chart.js code, and Mermaid.js syntax.\n",

### 🌐 Language Note

The `prebuilt-documentSearch` analyzer generates figure descriptions in **English by default**, regardless of the document's language. This is because:
- The `locales` parameter only applies to **audio/video** analyzers (for transcription)
- For documents, the LLM (GPT-4.1-mini) processes visual content and outputs in English

> 💡 **Tip**: If you need Hebrew descriptions, you can post-process the English descriptions using Azure OpenAI to translate them, or use the descriptions as-is since they're meant for RAG indexing (semantic search works across languages).

### ⏱️ Timing Note

Content Understanding takes longer than Document Intelligence because it:
- Detects each figure in the document
- Sends each figure to GPT-4.1-mini for AI description
- Analyzes charts/diagrams for structured output
- Generates formatted markdown

**Expect 3-10 minutes** depending on document size and figure count.

### 💡 Caching Strategy

In production, you'd cache CU results to avoid re-processing. We'll save results to a JSON file.

In [ ]:
# ⏱️ This cell analyzes the PDF with Content Understanding
# Expected time: 3-10 minutes (depends on figures and document size)
#
# 💡 TIP: If you want to skip the wait, run the NEXT cell instead
#    which loads pre-cached results.

CACHE_FILE = "metro_s36_cu_result.json"
analyzer_id = "prebuilt-documentSearch"

print(f"🔍 Analyzing {PDF_PATH.name} with Content Understanding...")
print(f"   Analyzer: {analyzer_id}")
print(f"   ⏱️  This may take 3-10 minutes. Please wait...\n")

# Read the PDF
with open(PDF_PATH, "rb") as f:
    file_bytes = f.read()

print(f"   File size: {len(file_bytes):,} bytes ({len(file_bytes)/1024/1024:.1f} MB)")

try:
    # Start analysis
    print("\n   Starting analysis...")
    
    if hasattr(cu_client, "begin_analyze_binary"):
        response = cu_client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=file_bytes,
            content_type="application/pdf"
        )
    else:
        response = cu_client.begin_analyze(
            analyzer_id=analyzer_id,
            body=file_bytes,
            content_type="application/pdf"
        )
    
    # Extract operation ID from the polling method's _async_url
    # URL format: .../analyzerResults/{operation_id}?api-version=...
    operation_id = ""
    try:
        if hasattr(response, '_polling_method'):
            pm = response._polling_method
            if hasattr(pm, '_operation'):
                op = pm._operation
                # The operation ID is in _async_url
                if hasattr(op, '_async_url') and op._async_url:
                    url = op._async_url
                    if '/analyzerResults/' in url:
                        operation_id = url.split('/analyzerResults/')[-1].split('?')[0]
    except Exception as e:
        print(f"   ⚠️ Could not extract operation ID: {e}")
    
    print("   Waiting for results (the LLM is describing each figure)...")
    result = response.result()
    
    print("\n✅ Analysis Complete!")
    print(f"   Operation ID: {operation_id}" if operation_id else "   Operation ID: (not available)")
    
    # Save to cache - include operation_id for figure downloads
    if hasattr(result, "as_dict"):
        result_dict = result.as_dict()
    else:
        result_dict = {"content": str(result)}
    
    # Store operation ID in cache for later use
    result_dict["_operation_id"] = operation_id
    result_dict["_analyzer_id"] = analyzer_id
    
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=2, ensure_ascii=False)
    
    print(f"   💾 Results cached to: {CACHE_FILE}")
    
    if not operation_id:
        print("\n   ⚠️ Note: Without operation ID, figure images cannot be downloaded.")
        print("   However, the AI descriptions are still embedded in the markdown!")

except Exception as e:
    print(f"\n❌ Analysis failed: {e}")
    print("\n   ⚠️ Try loading cached results in the next cell instead.")
    result_dict = None

In [ ]:
# ⏩ ALTERNATIVE: Load cached results (skip the 3-10 minute wait)
# Use this if you already ran the analysis before or want to skip waiting

CACHE_FILE = "metro_s36_cu_result.json"

# Initialize variables
markdown_text = ""
operation_id = ""
analyzer_id = "prebuilt-documentSearch"

if Path(CACHE_FILE).exists():
    print(f"📂 Loading cached results from: {CACHE_FILE}")
    
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        result_dict = json.load(f)
    
    # Get operation ID from cache (if saved)
    operation_id = result_dict.get("_operation_id", "")
    analyzer_id = result_dict.get("_analyzer_id", "prebuilt-documentSearch")
    
    print(f"   Operation ID: {operation_id[:50]}..." if operation_id else "   Operation ID: (not in cache - re-run analysis to enable figure downloads)")
    
    # Extract markdown from the result
    contents_list = result_dict.get("contents") or result_dict.get("result", {}).get("contents", [])
    
    if contents_list:
        markdown_text = contents_list[0].get("markdown", "")
        print(f"\n✅ Loaded {len(markdown_text):,} characters of markdown")
        print(f"   Found {markdown_text.count('!['):,} figure references")
        print(f"   Found {markdown_text.count('#'):,} markdown headers")
    else:
        # Fallback for different result structure
        markdown_text = result_dict.get("content", "")
        print(f"\n✅ Loaded {len(markdown_text):,} characters")
else:
    print(f"❌ Cache file not found: {CACHE_FILE}")
    print("   👉 Run the analysis cell above first.")

---

## Step 4: Inspect the Results - The "Invisible" is Now Visible!

Remember in Module 1, when we asked about the zoning map, our RAG pipeline had **no idea** what was in it?

Now let's see what Content Understanding extracted. The key breakthrough is:

**Figures now have AI-generated descriptions!**

The markdown format is:
```markdown
![ALT_TEXT](figures/PAGE.FIGURE_NUM "AI_SEMANTIC_DESCRIPTION")
```

For example:
```markdown
![Map legend](figures/1.1 "A zoning map showing an 800-meter radius around Metro Station 36...")
```

In [ ]:
# Let's see what CU extracted!

print("🔎 INSPECTING CONTENT UNDERSTANDING OUTPUT\n")
print("="*60)

if not markdown_text:
    print("❌ No markdown text available. Run the analysis or load cached results first.")
else:
    # Show the first part of the markdown
    print("📝 MARKDOWN OUTPUT (first 1500 chars):")
    print("-"*60)
    print(markdown_text[:1500])
    print("-"*60)
    print(f"\n... [{len(markdown_text):,} total characters]")

In [ ]:
# Now let's extract ALL figure descriptions to see what CU generated

print("🖼️ FIGURE DESCRIPTIONS EXTRACTED BY CONTENT UNDERSTANDING\n")

if markdown_text:
    # Pattern to match: ![alt](url "description")
    figure_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+?)(?:\s+"([^"]+)")?\)', re.DOTALL)
    figures = figure_pattern.findall(markdown_text)
    
    if figures:
        print(f"✅ Found {len(figures)} figure(s) with descriptions:\n")
        
        for i, (alt_text, url, description) in enumerate(figures, 1):
            print(f"{'='*60}")
            print(f"📷 FIGURE {i}")
            print(f"{'='*60}")
            print(f"   URL: {url}")
            print(f"   Alt-text: {alt_text[:80]}..." if len(alt_text) > 80 else f"   Alt-text: {alt_text}")
            
            if description:
                print(f"\n   🎉 AI SEMANTIC DESCRIPTION:")
                print(f"   \"{description[:500]}...\"" if len(description) > 500 else f"   \"{description}\"")
                print("\n   ✨ This description can now be SEARCHED and used by RAG!")
            else:
                print(f"\n   ⚠️ No semantic description (may be using layout-only mode)")
            print()
    else:
        print("⚠️ No figure tags found in markdown output.")
        print("   This may indicate the analyzer ran in layout-only mode.")
else:
    print("❌ No markdown text available.")

---

## Step 5: The Breakthrough - Comparing Module 1 vs Module 3

Let's see the dramatic difference between naive RAG and Content Understanding.

### The Question: "What types of land use surround Station 36?"

| Module 1 (Naive RAG) | Module 3 (Content Understanding) |
|----------------------|----------------------------------|
| Returns: `"מגורים א׳, תעסוקה, מסחר"` | Returns: Full AI description of the zoning map |
| Problem: Just disconnected Hebrew labels | Solution: Explains WHERE each zone is located |
| The MAP was invisible | The MAP is now described semantically |

In [ ]:
# Let's demonstrate the breakthrough with a specific example

print("🎯 THE BREAKTHROUGH: From 'Invisible' to 'Searchable'\n")
print("="*60)

# What naive RAG would have found (simulated)
naive_result = """
מגורים א׳
תעסוקה
מסחר ותעסוקה
מבני ציבור
שטחים פתוחים
"""

print(">>> MODULE 1 (Naive RAG) - What we found:")
print("-"*60)
print(naive_result)
print("-"*60)
print("❌ Problem: Just text labels extracted from the legend.")
print("   We have NO IDEA where these zones are on the map!")
print("   If someone asks 'What is east of the station?' we can't answer.")

print("\n\n>>> MODULE 3 (Content Understanding) - What we now have:")
print("-"*60)

# Search for map description in the CU output
if markdown_text:
    # Look for figure descriptions containing map-related content
    map_keywords = ["map", "zoning", "radius", "מפה", "אזור", "residential", "commercial"]
    
    figure_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+?)\s+"([^"]+)"\)', re.DOTALL)
    figures = figure_pattern.findall(markdown_text)
    
    map_description = None
    for alt, url, desc in figures:
        desc_lower = desc.lower()
        if any(kw in desc_lower for kw in map_keywords):
            map_description = desc
            break
    
    if map_description:
        print(f"\"{map_description}\"")
        print("-"*60)
        print("✅ SUCCESS! The AI now understands:")
        print("   - What the map shows (zoning around station)")
        print("   - What the colors represent (land use types)")
        print("   - Spatial relationships (what's north, south, east, west)")
        print("\n   🎉 This description is now SEARCHABLE by our RAG pipeline!")
    else:
        print("(Map description not found in cached results)")
        print("\nShowing example of what CU typically generates:")
        print('"A zoning map showing an 800-meter radius around Metro Station 36.')
        print('The map displays different land use zones including residential areas (pink),')
        print('commercial zones (blue), mixed-use areas, and public facilities.')
        print('The station is centrally located with residential neighborhoods')
        print('to the northeast and commercial districts to the southwest."')
else:
    print("(Run the analysis to see actual CU output)")

---

## Step 6: Download Figures and See the Full Markdown (Like CU Studio!)

Content Understanding stores figures in its internal storage. To display them like in CU Studio, we need to:
1. Download each figure using `get_result_file()`
2. Save them locally
3. Update the markdown to use local paths

Let's do this!

In [ ]:
# Download figures from CU and create a viewable markdown with images
import os

print("📥 DOWNLOADING FIGURES FROM CONTENT UNDERSTANDING\n")
print("="*70)

# Create figures directory
figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)

if not markdown_text:
    print("❌ No markdown text. Run the analysis or load cached results first.")
elif not operation_id:
    print("⚠️ No operation ID available in the cached result.")
    print("")
    print("   Why? The Content Understanding SDK doesn't always expose the")
    print("   operation ID needed to download figure images from CU storage.")
    print("")
    print("   Options:")
    print("   1. Re-run the analysis cell (cell 10) - the updated code may capture it")
    print("   2. Use Content Understanding Studio in Azure Portal to view images")
    print("   3. Continue with the text descriptions (still valuable for RAG!)")
    print("")
    print("="*70)
    print("📝 MARKDOWN OUTPUT WITH AI DESCRIPTIONS (images shown as placeholders):")
    print("="*70 + "\n")
    
    # The descriptions are still in the markdown - show them!
    display(Markdown(markdown_text[:5000]))
else:
    print(f"   Operation ID: {operation_id}")
    
    # Find all figure references in the markdown
    figure_refs = re.findall(r'figures/(\d+\.\d+)', markdown_text)
    unique_figures = sorted(list(set(figure_refs)))
    
    print(f"   Found {len(unique_figures)} unique figures to download\n")
    
    # Download each figure
    downloaded = 0
    local_markdown = markdown_text
    
    for fig_id in unique_figures:
        fig_path = f"figures/{fig_id}"
        local_path = figures_dir / f"{fig_id}.png"
        
        try:
            # Get the figure file from CU
            # Signature: get_result_file(operation_id, path) -> Iterator[bytes]
            fig_data_iter = cu_client.get_result_file(
                operation_id=operation_id,
                path=fig_path
            )
            
            # Collect all bytes from iterator
            fig_data = b''.join(fig_data_iter)
            
            # Save locally
            with open(local_path, "wb") as f:
                f.write(fig_data)
            
            # Update markdown to use local path
            local_markdown = local_markdown.replace(f"({fig_path}", f"({local_path}")
            
            downloaded += 1
            print(f"   ✅ Downloaded: {fig_path} ({len(fig_data):,} bytes)")
            
        except Exception as e:
            error_msg = str(e)[:100]
            print(f"   ⚠️ Could not download {fig_path}: {error_msg}")
    
    print(f"\n   Downloaded {downloaded}/{len(unique_figures)} figures")
    
    if downloaded > 0:
        # Save the updated markdown
        md_file = "metro_s36_cu_with_images.md"
        with open(md_file, "w", encoding="utf-8") as f:
            f.write(local_markdown)
        print(f"\n💾 Markdown with local images saved to: {md_file}")
        
        print("\n" + "="*70)
        print("👇 RENDERED MARKDOWN WITH IMAGES:")
        print("="*70 + "\n")
        
        # Display with images
        display(Markdown(local_markdown[:8000]))
    else:
        print("\n⚠️ Could not download figures. Showing markdown with descriptions:")
        display(Markdown(markdown_text[:5000]))

---

## 📊 Summary: What Content Understanding Adds

We've now completed the **extraction** phase of our RAG pipeline with three levels of sophistication:

| Module | Technology | Figures | Tables | Structure |
|--------|-----------|---------|--------|----------|
| **Module 1** | Naive (PyPDF) | ❌ Invisible | ❌ Destroyed | ❌ Lost |
| **Module 2** | Document Intelligence | ✅ Bounding boxes | ✅ Rows/columns | ✅ Roles |
| **Module 3** | Content Understanding | ✅ **AI descriptions** | ✅ Markdown tables | ✅ **Full markdown output** |

### 🎯 Key Takeaways

1. **Content Understanding = Document Intelligence + GPT-4.1-mini**
   - It builds on DI's structure extraction
   - Adds semantic understanding via LLM

2. **Figures are now searchable**
   - No longer just bounding boxes
   - AI-generated descriptions can be indexed and retrieved

3. **Complete markdown output**
   - Text + tables + figures combined in one structured document
   - Perfect for RAG indexing and LLM context

4. **Trade-off: Cost vs Quality**
   - CU costs more (LLM calls for each figure)
   - But the quality improvement is significant for visual documents

### 📦 What CU Gives You (Ready for Module 4)

Content Understanding provides the **raw material** for your RAG pipeline:
- Full markdown text (with headers for structure)
- Figure references with AI descriptions
- Tables in markdown format

**In Module 4**, you'll learn how to **chunk** this content effectively for search indexing.

---

## ➡️ What's Next?

| Module | What You'll Learn |
|--------|-------------------|
| **Module 4** | **Chunking Strategies** - How to split CU's markdown output into searchable chunks |
| **Module 5** | Azure AI Search - Indexing and hybrid search |
| **Module 6** | GraphRAG - Cross-document reasoning |

**Next**: [Module 4 – Chunking Strategies](../module-4-chunking/README.md)

---

## 🔍 Bonus: Why Did We Need Extra Code for Figure Downloads?

### The SDK vs REST API Difference

In **Content Understanding Studio**, getting the markdown with figures is seamless. But in our code, we had to extract the `operation_id` from SDK internals. Why?

**The official Python SDK (`azure-ai-contentunderstanding`) has a limitation:**
- It wraps the REST API response in a `LROPoller` object
- The `Operation-Location` header (containing the operation ID) is buried in internal attributes
- The `get_result_file()` method exists but requires the operation ID

**The Azure Samples repo uses a workaround:**
- They created a custom `AzureContentUnderstandingClient` wrapper class
- It uses raw REST API calls via `requests`
- The `Response` object preserves the `Operation-Location` header
- Figure downloads become trivial: `client.get_result_file(response, "figures/1.1")`

See: [Azure Samples - content_understanding_client.py](https://github.com/Azure-Samples/azure-ai-content-understanding-python/blob/main/python/content_understanding_client.py)

### Key Insight: You DON'T Need a Schema!

| What You Want | Analyzer | Schema Required? |
|---------------|----------|------------------|
| **General RAG** (markdown + figures for ANY PDF) | `prebuilt-documentSearch` | ❌ **No** |
| Extract specific fields (StationNumber, etc.) | Custom analyzer | ✅ Yes |

The `prebuilt-documentSearch` analyzer works out-of-the-box on any document - no schema needed!

In [ ]:
# 🎯 SIMPLER APPROACH: Using REST API directly (like Azure Samples does)
# This avoids the SDK's hidden operation_id issue
# Source: https://github.com/Azure-Samples/azure-ai-content-understanding-python

import requests
import time
from azure.identity import DefaultAzureCredential

class SimpleContentUnderstandingClient:
    """
    Simplified Content Understanding client using REST API.
    Based on Azure Samples: https://github.com/Azure-Samples/azure-ai-content-understanding-python
    """
    
    def __init__(self, endpoint: str, api_version: str = "2025-11-01"):
        self.endpoint = endpoint.rstrip("/")
        self.api_version = api_version
        self._credential = DefaultAzureCredential()
    
    def _get_headers(self):
        token = self._credential.get_token("https://cognitiveservices.azure.com/.default").token
        return {"Authorization": f"Bearer {token}"}
    
    def analyze_binary(self, analyzer_id: str, file_bytes: bytes, content_type: str = "application/pdf"):
        """Analyze binary content and return (result, response) tuple."""
        headers = self._get_headers()
        headers["Content-Type"] = content_type
        
        url = f"{self.endpoint}/contentunderstanding/analyzers/{analyzer_id}:analyzeBinary?api-version={self.api_version}"
        response = requests.post(url, headers=headers, data=file_bytes)
        response.raise_for_status()
        
        # Poll for completion
        operation_location = response.headers.get("Operation-Location")
        while True:
            poll_response = requests.get(operation_location, headers=self._get_headers())
            poll_response.raise_for_status()
            result = poll_response.json()
            
            if result.get("status") == "Succeeded":
                return result, response  # Return BOTH result and original response
            elif result.get("status") == "Failed":
                raise RuntimeError(f"Analysis failed: {result}")
            
            time.sleep(3)
    
    def get_result_file(self, analyze_response, file_id: str) -> bytes:
        """
        Get a result file (figure, keyframe, etc.) using the original response.
        The operation_id is extracted from the Operation-Location header.
        """
        # Extract operation ID from header - THIS IS THE KEY!
        operation_location = analyze_response.headers.get("Operation-Location", "")
        operation_id = operation_location.split("/analyzerResults/")[-1].split("?")[0]
        
        # Download the file
        url = f"{self.endpoint}/contentunderstanding/analyzerResults/{operation_id}/files/{file_id}?api-version={self.api_version}"
        response = requests.get(url, headers=self._get_headers())
        response.raise_for_status()
        return response.content

# Example usage (this is how Azure Samples does it!)
print("💡 SIMPLER APPROACH - Using REST API directly:")
print("-" * 60)
print("""
# 1. Create client
client = SimpleContentUnderstandingClient(endpoint)

# 2. Analyze (returns BOTH result and response)
result, response = client.analyze_binary("prebuilt-documentSearch", file_bytes)

# 3. Get markdown
markdown = result["contents"][0]["markdown"]

# 4. Download figures - EASY! No hidden operation_id dance!
figure_data = client.get_result_file(response, "figures/1.1")
""")
print("-" * 60)
print("✅ This is exactly how the Azure Samples repo does it!")
print("   See: github.com/Azure-Samples/azure-ai-content-understanding-python")

In [ ]:
# 🏗️ OPTIONAL: Custom Analyzers with Schemas
# Use this ONLY when you want to EXTRACT SPECIFIC FIELDS from known document types

print("📋 WHEN TO USE CUSTOM ANALYZERS vs PREBUILT")
print("=" * 60)
print("""
┌─────────────────────────────────────────────────────────────┐
│  YOUR USE CASE                      │  WHAT TO USE          │
├─────────────────────────────────────┼───────────────────────┤
│  Process ANY PDF for RAG            │  prebuilt-documentSearch  │
│  (markdown + figures, no schema)    │  ✅ NO SCHEMA NEEDED     │
├─────────────────────────────────────┼───────────────────────┤
│  Extract specific fields from       │  Custom Analyzer       │
│  known document types               │  ✅ SCHEMA REQUIRED    │
│  (e.g., invoice: total, vendor)     │                        │
│  (e.g., metro: StationNumber, Name) │                        │
└─────────────────────────────────────┴───────────────────────┘
""")

print("\n📝 The 'metro' analyzer you created in Studio with")
print("   StationNumber, StationName, StationType fields is for")
print("   STRUCTURED EXTRACTION - a different use case than RAG!")

print("\n" + "=" * 60)
print("EXAMPLE: Creating a Custom Analyzer via Code")
print("=" * 60)

# This is what you did in the Studio UI
custom_analyzer_example = {
    "description": "Extract metro station information",
    "fieldSchema": {
        "fields": {
            "StationNumber": {
                "type": "integer",
                "method": "extract",
                "description": "Unique station identifier"
            },
            "StationName": {
                "type": "string",
                "method": "extract", 
                "description": "Official station name"
            },
            "StationType": {
                "type": "string",
                "method": "extract",
                "description": "Station classification"
            }
        }
    }
}

print(json.dumps(custom_analyzer_example, indent=2, ensure_ascii=False))

print("\n" + "=" * 60)
print("🎯 BOTTOM LINE:")
print("=" * 60)
print("""
For your workshop (RAG with any PDF):
  → Use 'prebuilt-documentSearch' 
  → NO SCHEMA NEEDED
  → Works on any document!

The complexity we dealt with was:
  1. SDK limitation (hides operation_id) - not your fault!
  2. We solved it by extracting from internal attributes
  3. Azure Samples uses REST API wrapper to avoid this

The Studio makes it seamless because it uses REST API directly.
""")